# Init

In [ ]:
import copy
import importlib
import os
import pickle
import sys
import xml.etree.cElementTree as ET

import matplotlib.pyplot as plt
import mdtraj
# https://github.com/mellofariam/mmct
import mmct.mmct.force_field as ff
import mmct.mmct.macromolecule as mol
import mmct.mmct.pdb_tools as pdb_tools
import numpy as np
import pandas as pd
from scipy.spatial import distance

sys.path.append("../../")
from npc_ana_tool import *

In [2]:
# work_dir = '/home/ed31/Documents/LargeSystem/npc'
work_dir = '../../'

# Load data

In [3]:
df_c, top_c, xml_c = ff.load_force_field(
    pdb_file=f"{work_dir}/emin/constricted/reconstruct/constricted_reconstruct.pdb",
    top_file=f"{work_dir}/SBM/dual_basin/predual_constricted_CA.top",
    xml_file=f"{work_dir}/SBM/dual_basin/predual_constricted_CA.xml",
)

In [4]:
df_d, top_d, xml_d = ff.load_force_field(
    pdb_file=f"{work_dir}/emin/dilated/reconstruct/dilated_reconstruct.pdb",
    top_file=f"{work_dir}/SBM/dual_basin/predual_dilated_CA.top",
    xml_file=f"{work_dir}/SBM/dual_basin/predual_dilated_CA.xml",
)

In [5]:
df_chain_info

,new_id,model,start,end,CA_start,CA_end,Nup,Chain ID,Ring
0,00,7r5k,0,6084,0,755,Nup358 (RBP2),A,CR (unique)
1,01,7r5k,6085,12169,756,1511,Nup358 (RBP2),B,CR (unique)
2,02,7r5k,12170,18254,1512,2267,Nup358 (RBP2),C,CR (unique)
3,03,7r5k,18255,24339,2268,3023,Nup358 (RBP2),D,CR (unique)
4,04,7r5k,24340,30424,3024,3779,Nup358 (RBP2),E,CR (unique)
...,...,...,...,...,...,...,...,...,...
96,U4,7r5k,608641,608791,76965,76983,Nup98,SC,IR (FG)
97,U5,7r5k,608792,608942,76984,77002,Nup98,TC,IR (FG)
98,U6,7r5k,608943,609093,77003,77021,Nup98,UC,NR (FG)
99,V0,7r5k,609094,611296,77022,77294,Nup214,VC,CR (cytoplasmic filament)


In [6]:
# Rename the chain ids

count = 0
for i in range(len(df_chain_info)):
    start = df_chain_info.iloc[i]['CA_start']
    end = df_chain_info.iloc[i]['CA_end']
    for j in range(8):
        sub_start = start + j * N_subunit_CA 
        sub_end = end + j * N_subunit_CA
        df_c.loc[sub_start:sub_end+1, 'chain_id'] =  str(count)
        df_d.loc[sub_start:sub_end+1, 'chain_id'] =  str(count)
        count += 1

In [7]:
# Set up the trivial index mapping

dict_idx = dict()
for i in range(1, 1+N_subunit_CA*8):
    dict_idx[i] = i

# Dual basin model

In [ ]:
# this function captures the unsymmetric indices in the contact list, 
# represented with the first spoke.
from collections import Counter
def _get_unsymmetric_indices(df_contacts, i = 'i', j = 'j'):
    sorted_index_remainder = np.sort(df_contacts[[i, j]].values%N_subunit_CA, axis = 1)
    counts_remainder = Counter(map(tuple, sorted_index_remainder))
    unsymmetric = [item for item, count in counts_remainder.items() if count != 8]
    return unsymmetric

def define_multibasin_model_NPC(
    reference_top: dict[str, pd.DataFrame],
    reference_xml: ET.ElementTree,
    reference_pdb: pd.DataFrame,
    additional_top: dict[str, pd.DataFrame],
    additional_xml: ET.ElementTree,
    additional_pdb: pd.DataFrame,
    idx_from_additional_to_reference: dict[int, int],
    mode_angles: str = "middle",
    mode_contacts: str = "CG",
    xml_file: str = "smog.xml",
    top_file: str = "smog.top",
    contact_file: str = "contacts.pkl",
    scale_contact_distances: float = 1.0,
) -> tuple[
    dict[str, pd.DataFrame], ET.ElementTree, pd.DataFrame
]:
    """
    Updates the dihedrals and angles from the reference topology to the middle value between the two topologies.
    """

    multibasin_top = copy.deepcopy(reference_top)
    multibasin_xml = copy.deepcopy(reference_xml)

    # For the bonds
    print("Processing bonds...", flush=True, end="")
    multibasin_top, multibasin_xml = ff._process_bonds(
        reference_top,
        additional_top,
        multibasin_top,
        idx_from_additional_to_reference,
        multibasin_xml,
    )
    print(" Done.", flush=True)

    # For the angles

    print("Processing angles...", flush=True, end="")
    multibasin_top, multibasin_xml = ff._process_angles(
        reference_top,
        additional_top,
        multibasin_top,
        idx_from_additional_to_reference,
        multibasin_xml,
        mode_angles,
    )
    print(" Done.", flush=True)

    # For the dihedrals

    print("Processing dihedrals...", flush=True, end="")
    multibasin_xml = ff._process_dihedrals(
        reference_xml,
        additional_xml,
        multibasin_xml,
        idx_from_additional_to_reference,
    )
    print(" Done.", flush=True)

    # For the contacts

    print("Processing contacts...", flush=True, end="")
    multibasin_xml, df_contacts, merged_contacts = ff._process_contacts(
        reference_xml,
        reference_pdb,
        additional_xml,
        additional_pdb,
        multibasin_xml,
        idx_from_additional_to_reference,
        mode_contacts,
        scale_contact_distances,
        return_merged_contacts = True,
    )
    
    # make sure the unique contacts are symmetric, and update the A and B values for the unsymmetric contacts
    unsymmetric = _get_unsymmetric_indices(merged_contacts.query('source == "both"'))  
    update_lookup = dict()
    for pair in unsymmetric:
        sel = np.all(np.sort(merged_contacts[['i', 'j']].values%N_subunit_CA, axis = 1) == pair, axis = 1)
        A, B = merged_contacts.loc[sel].query('source == "both"')[['A', 'B']].mean()
        distance = merged_contacts.loc[sel].query('source == "both"')[['sigma_iso']].mean()
        
        df_contacts.loc[sel, 'source'] = 'common'
        df_contacts.loc[sel, 'distance'] = distance

        for i, j in merged_contacts.loc[sel].query('source != "both"')[['i', 'j']].values:
            update_lookup[(str(i), str(j))] = (A, B)


    for contact_type in multibasin_xml.findall(
            ".//contacts/contacts_type"
        ):
            for interaction in contact_type.findall("interaction"):

                xml_i = interaction.get('i')
                xml_j = interaction.get('j')

                if (xml_i, xml_j) in update_lookup:
                    A, B = update_lookup[(xml_i, xml_j)]
                    interaction.set('A', f"{A:.5e}")
                    interaction.set('B', f"{B:.5e}")
    

    multibasin_top = ff._update_exclusions(
        xml=multibasin_xml,
        top=multibasin_top,
    )
    print(" Done.", flush=True)

    # Save the new files

    print("Saving files...", flush=True, end="")

    ff.save_top(multibasin_top, top_file)
    ff.save_xml(multibasin_xml, xml_file)

    df_contacts.to_pickle(contact_file)

    print(" Done.", flush=True)

    return multibasin_top, multibasin_xml, df_contacts

In [9]:
top_dual, xml_dual, contacts_dual = define_multibasin_model_NPC(
    reference_top=top_c,
    reference_xml=xml_c,
    reference_pdb=df_c,
    additional_top=top_d,
    additional_xml=xml_d,
    additional_pdb=df_d,
    idx_from_additional_to_reference=dict_idx,
    mode_angles="flat_bottom",
    xml_file="dual_basin.xml.old",
    top_file="dual_basin.top",
    contact_file="dual_basin.pkl",
)

Processing bonds... Done.
Processing angles... Done.
Processing dihedrals...
 Done.
Processing contacts... Done.
Saving files... Done.


## Remove dihedrals of problematic angles

In [10]:
top_dual = ff.read_top("dual_basin.top")
xml_dual = ET.parse("dual_basin.xml.old")

In [11]:
xml_dual_new = ff.remove_unstable_dihedrals(xml_dual)
top_dual_new = copy.deepcopy(top_dual)

In [12]:
ff.save_xml(xml_dual_new, "dual_basin.xml")

# only common contacts

In [13]:
contacts_dual['source'].value_counts()

source
common    2050448
left        49248
right       35680
Name: count, dtype: int64

In [14]:
atom_pairs_unique = contacts_dual.query(
        "source in ['left', 'right']"
    )[["i", "j"]].values

In [15]:
# From the original force field, I'll delete the unique contacts
xml_dual_common, top_dual_common = ff.delete_contacts(
    xml_dual_new,
    top_dual_new,
    atom_pairs = atom_pairs_unique,
)

In [16]:
ff.save_xml(xml_dual_common, f"dual_basin_common.xml")
ff.save_top(top_dual_common, f"dual_basin_common.top")

# Write unique contacts

In [17]:
from Contacts.Contacts import *

In [18]:
contact_DB_uniq_c = ContactsOnuchic() # left
contact_DB_uniq_d = ContactsOnuchic() # right

In [20]:
contacts_left = contacts_dual.query("source == 'left'")

In [21]:
contact_DB_uniq_c.addContacts(
    contacts_left['i'].values - 1, # convert to 0-based index
    contacts_left['j'].values - 1, # convert to 0-based index
    contacts_left['distance'].values,
)

In [22]:
contacts_right = contacts_dual.query("source == 'right'")

In [23]:
contact_DB_uniq_d.addContacts(
    contacts_right['i'].values - 1,
    contacts_right['j'].values - 1,
    contacts_right['distance'].values,
)

In [24]:
with open("contact_DB_uniq_c.pkl", "wb") as f:
    pickle.dump(contact_DB_uniq_c, f)

with open("contact_DB_uniq_d.pkl", "wb") as f:
    pickle.dump(contact_DB_uniq_d, f)